# RAG — Retrieval-Augmented Generation

Fine-tuning and RAG are the two ways to adapt a pretrained model. **Fine-tuning changes the weights.
RAG leaves the model frozen and puts the knowledge into the prompt**, looked up at question time.

Related notebooks: [Fine-tuning Vision Transformers](../transfer_learning_fine_tuning/vit_fine_tuning.ipynb) ·
[llama.cpp, GGUF & quantization](../llama_cpp_gguf/index.ipynb) ·
[Cross-modal retrieval](../multimodal_models/cross_modal_retrieval.ipynb) ·
[DINOv3](../foundation_models/DINOv3.ipynb)

Runnable companion script: [`scripts/mini_rag.py`](scripts/mini_rag.py) — a whole RAG pipeline
(chunk → embed → index → retrieve → rerank → prompt) in ~150 lines of PyTorch/NumPy, no framework.


## 1. Fine-tuning vs RAG

| | Fine-tuning | RAG |
| --- | --- | --- |
| What changes | the model's weights: all, some, or LoRA adapters | nothing; the model stays frozen |
| Where new knowledge goes | into the weights, through training | into the prompt: relevant documents are looked up at question time and pasted into the context |
| Typical models | vision models: DINOv3, RF-DETR, YOLO | language and vision-language models: an LLM, Qwen3-VL |
| To update knowledge | retrain | add documents to the index |
| Cost per update | GPU-hours | seconds |
| Failure mode | catastrophic forgetting, stale knowledge | retrieval misses, context dilution |
| Provenance | none — you cannot point at the source of an answer | built in — every answer carries its retrieved chunks |

**Rule of thumb:** RAG is how you adapt an LLM or VLM *without* training it; fine-tuning is how you
adapt a vision model *by* training it. They compose: fine-tune (or LoRA) for **behaviour, format and
domain vocabulary**, use RAG for **facts that change**.


## 2. The pipeline

```mermaid
flowchart LR
  subgraph Offline
    D[Documents] --> C[Chunking]
    C --> E1[Embedding model<br/>frozen encoder]
    E1 --> I[(Vector index)]
  end
  subgraph Online
    Q[Question] --> E2[Same encoder]
    E2 --> S[Top-k search<br/>dense + BM25]
    I --> S
    S --> R[Reranker<br/>cross-encoder]
    R --> P[Prompt:<br/>context + question]
    P --> L[Frozen LLM]
    L --> A[Answer + citations]
  end
```

Two properties of this picture are worth naming, because they are exactly the ones from §3 of the
fine-tuning notebook:

* The **encoder is frozen** — just like a frozen ViT backbone. You are reusing general pretrained
  features rather than training new ones.
* The only *learned* part of a typical RAG system is the reranker, and usually you don't train that
  either — you download one.


## 3. Chunking

Retrieval quality is bounded by chunking quality; this is where most RAG systems are actually lost.

* **Split on structure first** (headings, functions, table rows), on tokens second. A chunk that starts
  mid-sentence embeds badly.
* **Target 200–500 tokens** per chunk for prose, with **10–20 % overlap** so an answer that straddles a
  boundary survives.
* **Keep a header trail** in the chunk text (`file > section > subsection`). Embeddings are
  context-free: "the default is 0.8" is unretrievable without knowing what "the" is.
* **Store metadata** next to the vector: source path, line range, date, permissions. You need it for
  citations, for filtering, and for deleting a document from the index later.

```python
def chunk(text, size=400, overlap=60):
    words = text.split()
    step = size - overlap
    return [" ".join(words[i:i + size]) for i in range(0, max(len(words), 1), step)]
```

For code, chunk per function/class with the file path and imports prepended; for tables, one chunk per
row group with the header row repeated.


## 4. Embeddings and the index

An embedding model maps a chunk to a unit vector so that cosine similarity ≈ semantic similarity. Since
the vectors are L2-normalised, cosine similarity is just a dot product, and top-k search is one matmul:

```python
import torch

emb = torch.nn.functional.normalize(emb, dim=-1)       # (N, d)
q   = torch.nn.functional.normalize(q,   dim=-1)       # (1, d)
scores = q @ emb.T                                     # (1, N)
top = scores.topk(k, dim=-1)
```

For up to ~100 k chunks this brute-force matmul is fast enough on a GPU and exact — reach for an ANN
index (FAISS `IndexHNSWFlat`, `IndexIVFPQ`, or a vector database) only past that.

| Choice | Typical | Notes |
| --- | --- | --- |
| Embedding model | `BAAI/bge-m3`, `Qwen3-Embedding-0.6B`, `intfloat/multilingual-e5-large` | run locally; 0.5–1 B params is plenty |
| Dimension | 768–1024 | 1 M chunks × 1024 dims × fp16 = 2 GB |
| Distance | cosine (= dot on normalised vectors) | normalise once, at index time |
| Prefixes | `query: …` / `passage: …` for E5-style models | using the wrong prefix quietly costs several points of recall |
| Quantisation | fp16, or int8 with a re-score of the top 100 | halves the index, costs ~1 % recall |

**Symmetry matters:** questions and documents must be embedded by the *same* model, with the model's
own instruction prefixes. This is the RAG equivalent of "use ImageNet normalisation for a pretrained
ViT" from the fine-tuning checklist — get it wrong and everything still runs, just worse.


## 5. Persisting the index: Chroma

The matmul of §4 is exact and fast, but it is rebuilt on every start and knows nothing about
metadata. A vector store fixes both. [Chroma](https://www.trychroma.com/) is the smallest one that is
still a real store — an embedded SQLite + HNSW index, one `pip install`, no server:

```bash
pip install chromadb
```

```python
import chromadb

client = chromadb.PersistentClient(path="./chroma_rag")
col = client.get_or_create_collection("rag_corpus",
                                      metadata={"hnsw:space": "cosine"})   # not the default L2

col.upsert(ids=ids,                       # stable ids: a content hash per chunk
           embeddings=vectors.tolist(),   # (N, d), already L2-normalised
           documents=texts,
           metadatas=[{"source": c["source"], "header": c["header"]} for c in chunks])

r = col.query(query_embeddings=qvec.tolist(), n_results=50,
              where={"source": {"$contains": "transfer_learning"}},   # filter BEFORE the search
              include=["documents", "metadatas", "distances"])
score = 1.0 - r["distances"][0][0]        # cosine distance -> similarity
```

Four things to get right:

* **`hnsw:space`.** Chroma defaults to squared L2. With normalised vectors the ranking happens to
  agree, but the scores do not — set `cosine` at creation time; it cannot be changed afterwards.
* **Stable ids.** Use a hash of the chunk text, not `enumerate()`. Then re-indexing an unchanged file
  is a no-op and an edited file overwrites its own chunks instead of duplicating them:

  ```python
  chunk_id = hashlib.sha1(chunk["text"].encode()).hexdigest()[:16]
  known = set(col.get(ids=ids, include=[])["ids"])      # upsert only what is new
  ```
* **Metadata filters run before the ANN search**, which is what makes permission filtering correct —
  post-filtering a top-k can return fewer results than asked, or none.
* **The collection belongs to one embedding model.** Vectors from a different model are not
  comparable; swapping the embedder means rebuilding the collection (§11 pitfall table).

`chromadb` ships its own default embedding function (ONNX MiniLM) if you pass `documents` without
`embeddings`. Convenient for a demo, but explicit is better: it silently pins you to that model.

In the companion script this is one flag:

```bash
python rag/scripts/mini_rag.py --store chroma --persist ./chroma_rag \
    --corpus . --ask "how does layer-wise LR decay work?"
# chroma: +7 new chunks, 7 in collection at ./chroma_rag   (second run: +0)
```

**When to move past Chroma:** it is an embedded library, so it scales with one process and one disk.
Past a few million chunks, or once several services write to the index, go to FAISS with your own
storage layer, or a served store (Qdrant, Weaviate, pgvector). The retrieval code above does not change.


## 6. Hybrid retrieval and reranking

Dense embeddings miss exact tokens: error codes, flags, version numbers, rare names. BM25 (sparse,
lexical) catches those and misses paraphrases. Run both and fuse — Reciprocal Rank Fusion is one line
and needs no score calibration:

```python
def rrf(ranked_lists, k=60):
    scores = {}
    for lst in ranked_lists:                       # each: doc ids, best first
        for rank, doc in enumerate(lst):
            scores[doc] = scores.get(doc, 0.0) + 1.0 / (k + rank + 1)
    return sorted(scores, key=scores.get, reverse=True)
```

Then **rerank**: take the fused top-50 and score each (question, chunk) pair with a cross-encoder
(`BAAI/bge-reranker-v2-m3`). A bi-encoder embeds the two texts separately; a cross-encoder reads them
together and is far more accurate — and far slower, which is why it only ever sees the top-50, never the
whole index.

Rule of thumb for the budget: retrieve 50, rerank to 5–8, put those in the prompt. More context is not
better — precision at the top of the list is what the generator actually uses.


## 7. The prompt, and grounding the answer

```
Answer the question using ONLY the context below.
If the context does not contain the answer, say you don't know.
Cite the [n] of every chunk you use.

[1] (docs/training.md > LLRD) The patch embeddings move about 30x more slowly ...
[2] (docs/training.md > Schedule) warm-up 0.1 epoch, cosine to 5% ...

Question: how is the learning rate spread over the backbone?
```

* **Number the chunks** and demand citations — it makes hallucinations visible instead of plausible.
* **Give an explicit escape hatch** ("say you don't know"), or the model will invent an answer when
  retrieval fails.
* **Order matters**: models attend most to the beginning and end of a long context ("lost in the
  middle"), so put the best chunk first and the question last.
* **Keep the context tight.** Ten mediocre chunks are worse than three good ones.


## 8. Generating with a local model (Ollama)

Everything so far is retrieval; the generator is the frozen part. A quantised 30B MoE runs the answer
step comfortably on one 24 GB card:

```bash
ollama pull qwen3:30b-a3b-instruct-2507-q4_K_M    # generator, ~18 GB on disk
ollama pull qwen3-embedding:0.6b                  # embedder, 1024-d vectors
```

`qwen3:30b-a3b` is a mixture of experts: 30.5 B total parameters, ~3 B active per token. All the weights
must sit in VRAM, but only the active experts are multiplied, so it reads like a 3 B model and answers
like a much larger one — the reason it is a good RAG generator on consumer hardware. See
[llama.cpp, GGUF & quantization](../llama_cpp_gguf/index.ipynb) for what `q4_K_M` means.

The whole client is two HTTP calls to `localhost:11434`, no SDK:

```python
# embeddings (offline indexing and the query)
r = requests.post("http://localhost:11434/api/embed",
                  json={"model": "qwen3-embedding:0.6b", "input": texts}).json()
vectors = np.asarray(r["embeddings"], dtype=np.float32)
vectors /= np.linalg.norm(vectors, axis=1, keepdims=True)

# generation
r = requests.post("http://localhost:11434/api/chat", json={
        "model": "qwen3:30b-a3b-instruct-2507-q4_K_M",
        "messages": [{"role": "user", "content": prompt}],
        "stream": False,
        "options": {"num_ctx": 8192, "temperature": 0.0}}).json()
answer = r["message"]["content"]
```

### The four settings that matter

* **`num_ctx` — the one that silently ruins RAG.** Ollama's default context is 4096 tokens
  (`OLLAMA_CONTEXT_LENGTH`), *not* the model's 262 144. A prompt longer than that is truncated from the
  front — which is exactly where your retrieved context sits — and nothing warns you. Set `num_ctx` to
  cover the prompt: roughly `k × chunk_tokens × 1.4 + question + answer`. For k = 5 chunks of 400 words,
  8192 is comfortable.
* **KV cache is the other VRAM cost.** Context memory grows linearly with `num_ctx`, on top of the 18 GB
  of weights. On a 24 GB card keep `num_ctx` at 8–16 k for this model; past that, layers spill to CPU and
  the token rate collapses.
* **`temperature: 0`** for RAG. You want the retrieved text repeated faithfully, not paraphrased
  creatively. Qwen3's own default is 0.7.
* **Two models, one GPU.** The embedder (~1.5 GB) and the generator (~18 GB) both stay resident with the
  default 5-minute keep-alive, which fits. If you see them swapping in and out on every query, pin them
  with `OLLAMA_KEEP_ALIVE=1h` or index once and only load the generator afterwards.

### Query-side instructions

Qwen3-Embedding is trained with an instruction on the **query** side and none on the passage side —
the same asymmetry as E5's `query:`/`passage:` prefixes (§4):

```python
QUERY = "Instruct: Given a question, retrieve passages that answer it\nQuery: "
qvec = embed([QUERY + question])        # documents are embedded unprefixed
```

Skipping it costs a few points of recall, silently. `mini_rag.py` applies it automatically when the
embedding model name contains `qwen3-embedding`.

### End to end

```bash
python rag/scripts/mini_rag.py --corpus rag \\
    --ask "Why does the Ollama default context length break RAG?" \\
    --k 3 --no-rerank --generate
```

```text
23 chunks from rag
embedder: ollama qwen3-embedding:0.6b

retrieved:
  [1] index.ipynb > RAG ... > 8. Generating with a local model (Ollama) > The four settings that matter
  [2] scripts/mini_rag.py
  [3] index.ipynb > RAG ... > 13. Further reading

generating with qwen3:30b-a3b-instruct-2507-q4_K_M (num_ctx 8192) ...
prompt 1749 tok -> 90 tok
======================================================================
The Ollama default context length of 4096 tokens breaks RAG because it is much smaller than the
model's actual context length (262,144 tokens). When a prompt exceeds this limit, it gets truncated
from the front — which is exactly where the retrieved context sits — and no warning is issued. [1]
```

Five seconds end to end on one RTX 3090, and the answer carries the chunk it came from.

The script picks the first embedding backend that answers — Ollama, then sentence-transformers, then a
TF-IDF fallback — so it runs with or without a model server. It indexes `.md`, `.py`, `.txt` and
`.ipynb` (notebooks are flattened to markdown, outputs dropped), which is what makes a repository of
notebooks like this one searchable at all.

**Indexing dominates the wall clock.** A query is one embed call plus one generation; embedding a whole
repository is thousands of them — scoping `--corpus` to a subdirectory is the difference between
seconds and a coffee break. That asymmetry is the whole argument for the persistent Chroma collection
of §5: index once, query forever.


## 9. Evaluating a RAG system

Measure the two halves separately, or you will tune the wrong one.

| Stage | Metric | What it tells you |
| --- | --- | --- |
| Retrieval | recall@k, MRR, nDCG | is the answer even in the context? |
| Reranking | precision@5 | is it near the top? |
| Generation | faithfulness (is every claim supported by a cited chunk), answer relevance | did the model use what it was given? |
| End to end | exact match / LLM-as-judge on a held-out question set | the only number worth quoting |

Build a small gold set: 50–100 real questions with the chunk id that answers each. It takes an afternoon
and turns every later change into a measurement rather than a debate. Same discipline as the held-out
split in the fine-tuning checklist — freeze it before you start tuning.

If recall@50 is low, the fix is in chunking or the embedding model; if recall@50 is high but the answers
are wrong, the fix is in the reranker or the prompt.


## 10. Visual RAG — retrieval with a frozen vision backbone

The retrieval half of RAG is exactly the frozen backbone of the fine-tuning notebook, used for
similarity instead of for a head.

**k-NN classification with frozen DINOv3 embeddings.** Embed every labelled crop once with a frozen
DINOv3 (the CLS token, L2-normalised), then label a new crop by its nearest neighbours. No training,
no head, and new classes cost one forward pass per example rather than a retraining run:

```python
bank = F.normalize(embed(labelled_crops), dim=-1)      # (N, d), computed once
q    = F.normalize(embed(new_crop),       dim=-1)      # (1, d)
sim, idx = (q @ bank.T).topk(k=5, dim=-1)
pred = majority_vote(labels[idx], weights=sim)
```

This is retrieval without the generation step, and it is the right baseline before *any* fine-tune:
it tells you how separable your classes already are in the frozen feature space.

**Visual RAG for a VLM.** A vision-language model used zero-shot ("is there a broken roof tile here?")
can be improved with no training by retrieving the 3–5 most similar *confirmed* examples from your own
archive and putting them in the prompt as labelled exemplars: "these are broken roof tiles — is there
one in this photo?". Same mechanism as text RAG, with an image encoder doing the retrieval.

The related idea in pure vision is **visual prompting** (e.g. YOLOE): the detector is conditioned on
example images instead of retrained. In one defect benchmark it scored 0.563 against a fine-tuned
model's 0.648 over 8 classes — clearly behind a real fine-tune, but available the same afternoon and
extensible by adding examples.


## 11. Pitfalls checklist

| Pitfall | Symptom | Fix |
| --- | --- | --- |
| Query/passage prefixes mismatched | recall quietly a few points low | use the model card's prefixes on both sides |
| Chunks lack their heading trail | retrieves the right document, wrong section | prepend `file > section` to the chunk text |
| Dense-only retrieval | error codes and flags never found | add BM25 + RRF |
| Too much context | the model answers from the wrong chunk | rerank, cut to 5–8 chunks |
| Index never updated | confidently stale answers | re-embed on change; store a content hash per chunk |
| No escape hatch in the prompt | fluent invented answers | "if the context does not answer it, say so" |
| No gold set | every change is an opinion | 50 questions + expected chunk ids |
| Embedding model swapped without re-indexing | garbage similarities | the whole index must be rebuilt; the vectors are not comparable across models |
| Permissions ignored | users retrieve documents they may not read | filter by metadata *before* the top-k, not after |
| `num_ctx` left at the Ollama default (4096) | the answer ignores the context that was retrieved for it | set `num_ctx` to cover the prompt (§8) |
| No query-side instruction for Qwen3-Embedding / E5 | a few points of recall, silently | prefix the query, not the passages (§4, §8) |


## 12. When to fine-tune anyway

RAG cannot fix everything. Fine-tune (or LoRA) when:

* the required **output format or behaviour** is unusual — structured JSON, a house style, tool-calling
  conventions. No amount of context reliably buys this;
* the domain has **its own vocabulary** that the base model tokenises poorly;
* **latency or cost** rules out a long context on every request — knowledge baked into the weights is
  free at inference time;
* the task is **perceptual rather than factual** — this is the vision case, where fine-tuning is simply
  the tool (see [Fine-tuning Vision Transformers](../transfer_learning_fine_tuning/vit_fine_tuning.ipynb)).

The common production shape is both: a LoRA-tuned model for behaviour, RAG for facts, and a frozen
embedding model shared between them.


## 13. Further reading

- [RAG (Lewis et al., 2020)](https://arxiv.org/abs/2005.11401) — the original paper
- [Lost in the Middle](https://arxiv.org/abs/2307.03172) — why context order matters
- [BGE-M3](https://huggingface.co/BAAI/bge-m3) and [bge-reranker-v2-m3](https://huggingface.co/BAAI/bge-reranker-v2-m3) — solid local embedder + reranker
- [MTEB leaderboard](https://huggingface.co/spaces/mteb/leaderboard) — pick an embedding model by task, not by hype
- [FAISS](https://github.com/facebookresearch/faiss) — when brute force stops being enough
- [RAGAS](https://github.com/explodinggradients/ragas) — faithfulness / relevance metrics
- [ColPali](https://arxiv.org/abs/2407.01449) — retrieval over page *images* with a VLM, no OCR
